# 🟡 Interview: SwiGLU Activation

Interview solution for the SwiGLU activation function.

$$\text{SwiGLU}(x) = (xW_1) \cdot \text{Swish}(xW_{gate}) \cdot W_2$$

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def swiglu(x, W1, W2, Wgate):
    """
    手写 SwiGLU —— 面试版。

    面试考点:
    1. Swish = z * sigmoid(z)，不要用 F.silu()，手写更显功力
    2. GLU 的本质是「门控」：一个分支做信号，另一个分支做门
    3. 两路投影 + 逐元素乘 + 下投影，三步缺一不可

    参数:
        x: (B, d_model) 输入
        W1: (d_model, d_ff) 上投影
        W2: (d_ff, d_model) 下投影
        Wgate: (d_model, d_ff) 门控投影

    返回: (B, d_model)
    """
    # ---- Step 1: 两路线性投影 ----
    # h 和 gate_raw 都是 (B, d_ff)，但走不同的权重矩阵
    # h 是「值」路径，gate_raw 是「门控」路径
    h = x @ W1              # (B, d_model) @ (d_model, d_ff) = (B, d_ff)
    gate_raw = x @ Wgate    # (B, d_ff)

    # ---- Step 2: Swish 激活（手写，不用 F.silu） ----
    # Swish(z) = z * sigmoid(z)
    # sigmoid(z) = 1 / (1 + exp(-z))
    # 面试时直接写 torch.sigmoid(z) * z 即可，不要调 F.silu
    swish_gate = gate_raw * torch.sigmoid(gate_raw)  # (B, d_ff)

    # ---- Step 3: 门控 ----
    # 逐元素相乘：值路径 * 门控路径
    # 这就是 Gated Linear Unit 的核心
    hidden = h * swish_gate  # (B, d_ff)

    # ---- Step 4: 下投影 ----
    return hidden @ W2  # (B, d_ff) @ (d_ff, d_model) = (B, d_model)

In [ ]:
# Verify
torch.manual_seed(0)
B, D, H = 2, 8, 16
x = torch.randn(B, D)
W1 = torch.randn(D, H)
W2 = torch.randn(H, D)
Wgate = torch.randn(D, H)

out = swiglu(x, W1, W2, Wgate)
print("Output shape:", out.shape)

gate = x @ Wgate
swish_gate = gate * torch.sigmoid(gate)
print("Gate (swish) sample values:", swish_gate[0, :4])

In [ ]:
from torch_judge import check
check("swiglu")

## 📝 核心思路总结

1. **SwiGLU = Swish + GLU**：两条并行的线性投影路径，一条产生值 (h)，一条经过 Swish 激活产生门控信号，逐元素相乘后再下投影。
2. **Swish vs ReLU**：`Swish(z) = z * sigmoid(z)` 在负值区域有小的非零输出，梯度更平滑，训练效果优于 ReLU。
3. **门控机制**：GLU 的核心思想是让网络自己学习「哪些维度该保留、哪些该抑制」，比固定的激活函数更灵活。
4. **参数量**：SwiGLU 有 3 个权重矩阵（W1, Wgate, W2），比标准 MLP 多一个，所以 d_ff 通常设为 `(2/3) * 4 * d_model` 来保持总参数量不变。